In [ ]:
# SHAM_SECRETS_FALLBACK_V2 (2026-10-15)
# كاكل يرفض قراءة الأسرار (HTTP 400) في النسخ المرفوعة آلياً من المراقب.
# لا تغيّر شيئاً حين تعمل الأسرار. وإلا: GITHUB_TOKEN -> استنساخ عام بلا توكن،
# KAGGLE_* -> kaggle.json من مجموعة البيانات الخاصة sham-runtime-creds (تُضبط
# في متغيرات البيئة فوراً كي تجدها كل الوحدات)، وتيليجرام يبقى اختيارياً.
def _sham_secrets_fallback_v2():
    import glob, json, os, subprocess
    os.environ.setdefault("GIT_TERMINAL_PROMPT", "0")
    subprocess.run(["git", "config", "--global", "url.https://github.com/.insteadOf", "https://@github.com/"], check=False)

    def creds():
        for p in sorted(glob.glob("/kaggle/input/**/kaggle.json", recursive=True)):
            try:
                with open(p) as f:
                    d = json.load(f)
                if d.get("username") and d.get("key"):
                    return d
            except Exception:
                pass
        return {}

    c = creds()
    print("sham-runtime-creds:", "موجودة" if c else "غير موجودة في المدخلات")
    if c and not (os.environ.get("KAGGLE_USERNAME") and (os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN"))):
        os.environ["KAGGLE_USERNAME"] = c["username"]
        os.environ["KAGGLE_KEY"] = c["key"]
        if c["key"].startswith("KGAT_"):
            os.environ["KAGGLE_API_TOKEN"] = c["key"]
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception:
        return
    if getattr(UserSecretsClient.get_secret, "_sham_safe2", False):
        return
    orig = UserSecretsClient.get_secret

    def get_secret(self, label, *a, **k):
        try:
            return orig(self, label, *a, **k)
        except Exception:
            if label == "GITHUB_TOKEN":
                print("تنبيه: سرّ GITHUB_TOKEN غير متاح لهذا التشغيل — استنساخ عام بلا توكن.")
                return ""
            if label in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
                d = creds()
                if d:
                    return d["username"] if label == "KAGGLE_USERNAME" else d["key"]
            raise

    get_secret._sham_safe2 = True
    UserSecretsClient.get_secret = get_secret

try:
    _sham_secrets_fallback_v2()
except Exception as _sf2_exc:
    print("SHAM_SECRETS_FALLBACK_V2 غير مفعّل:", type(_sf2_exc).__name__)


In [ ]:
# SHAM_SECRETS_FALLBACK_V3 (2026-10-15)
# بعد ساعات من التدريب فشلت خلية الحفظ بـ NameError: '_sf_orig' / '_sf_creds'
# (الغلاف القديم V1 يعتمد على متغيرات عامة اختفت). هذه الخلية مستقلة تماماً:
# لا تعتمد على أي متغير عام، تضبط KAGGLE_USERNAME/KAGGLE_KEY في البيئة و
# ~/.kaggle/kaggle.json فوراً، وتلتقط أي خطأ من الأغلفة السابقة (حتى NameError)
# فترجع القيم من البيئة. آمنة إن أُعيد تشغيلها.
def _sham_secrets_fallback_v3():
    import glob, json, os

    def load_creds(_glob=glob, _json=json, _os=os):
        u, k = _os.environ.get("KAGGLE_USERNAME"), _os.environ.get("KAGGLE_KEY") or _os.environ.get("KAGGLE_API_TOKEN")
        if u and k:
            return {"username": u, "key": k}
        pats = ["/kaggle/input/sham-runtime-creds/**/kaggle.json", "/kaggle/input/**/kaggle.json",
                _os.path.expanduser("~/.kaggle/kaggle.json")]
        for pat in pats:
            for p in sorted(_glob.glob(pat, recursive=True)):
                try:
                    with open(p) as f:
                        d = _json.load(f)
                    if d.get("username") and d.get("key"):
                        return d
                except Exception:
                    pass
        return {}

    c = load_creds()
    if c:
        os.environ.setdefault("KAGGLE_USERNAME", c["username"])
        if not (os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN")):
            os.environ["KAGGLE_KEY"] = c["key"]
            if str(c["key"]).startswith("KGAT_"):
                os.environ["KAGGLE_API_TOKEN"] = c["key"]
        try:
            kp = os.path.expanduser("~/.kaggle/kaggle.json")
            if not os.path.exists(kp):
                os.makedirs(os.path.dirname(kp), exist_ok=True)
                with open(kp, "w") as f:
                    json.dump({"username": c["username"], "key": c["key"]}, f)
                os.chmod(kp, 0o600)
        except Exception:
            pass
    print("SHAM_SECRETS_FALLBACK_V3: بيانات كاكل", "جاهزة في البيئة" if c else "غير موجودة")
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception:
        return
    if getattr(UserSecretsClient.get_secret, "_sham_safe3", False):
        return

    def get_secret(self, label, *a, _prev=UserSecretsClient.get_secret, _load=load_creds, **k):
        try:
            return _prev(self, label, *a, **k)
        except BaseException as e:
            if isinstance(e, KeyboardInterrupt):
                raise
            if label == "GITHUB_TOKEN":
                return ""
            if label in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
                d = _load()
                if d:
                    return d["username"] if label == "KAGGLE_USERNAME" else d["key"]
            raise

    get_secret._sham_safe3 = True
    get_secret._sham_safe2 = True  # يمنع V2 من إعادة التغليف لو أعيد تشغيلها
    get_secret._sham_safe = True   # ويمنع V1 كذلك
    UserSecretsClient.get_secret = get_secret

try:
    _sham_secrets_fallback_v3()
except Exception as _sf3_exc:
    print("SHAM_SECRETS_FALLBACK_V3 غير مفعّل:", type(_sf3_exc).__name__)


# شام — دفتر المحادثة والمرمزات (المرحلة الثالثة + ترميز الصورة ثم الصوت والفيديو)

دفتر واحد يحوي **كل خلايا** دفترين: المرحلة الثالثة — الدمج والمحادثة ومسار ترميز الصورة (ثم الصوت والفيديو). كل قسم خلاياه كما هي في دفتره الأصلي (مرئية وقابلة للتعديل)، ويعمل القسمان في الجلسة نفسها بالتتابع؛ ميزانية وقت كل قسم هنا مخفّضة (ست ساعات للمحادثة على CPU بدل 11) كي يتسع الاثنان في جلسة واحدة (حدّها 12 ساعة).

- **المسرّع:** CPU (أو GPU إن شئتِ؛ كاكل لا يجدول دفتراً عليه GPU).
- **Internet:** يجب أن يكون مفعّلاً. **الأسرار:** KAGGLE_USERNAME و KAGGLE_KEY و GITHUB_TOKEN. لا يحتاج أي Add Input.
- **الاستئناف:** كل قسم يستأنف تلقائياً من آخر نقطة منشورة.
- يُدار بمراقب إعادة التشغيل التلقائي (يعيد تشغيله بعد توقفه بمهلة قصيرة)، فلا تُجدوِليه من كاكل.
- لتشغيل قسم بعينه يدوياً: غيّري السطر `FORCE_SECTIONS` في خلية الاختيار.


## 1) سحب الكود الحقيقي من GitHub

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
    print("تم سحب الكود الحقيقي من مستودع GitHub بنجاح.")
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
    print("الكود موجود بالفعل في هذه الجلسة — تم سحب أي تحديثات جديدة عليه.")

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py")), f"لم يتم العثور على model.py داخل {CODE_DIR}"
sys.path.insert(0, CODE_DIR)
print("كود ShamSmall الحقيقي جاهز في:", CODE_DIR)


## 2) تثبيت المكتبات

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "av", "soundfile", "tokenizers", "datasets", "kaggle"], check=False)
print("تم تجهيز المكتبات.")


## 3) ما يعمل اليوم

In [ ]:
import datetime
import os

import torch

import sham_inputs  # يثبّت إصلاحات التدريب المشتركة ويطبّق تحديثات الخلايا
from sham_chain import STAGE_GROUPS, install_section_gate, pick_stages

CUDA = torch.cuda.is_available()
FORCE_SECTIONS = []   # مثال: ["sham_chat_stage"]  — الأقسام: sham_chat_stage, sham_image_tokenizer_track
STAGES_TODAY = FORCE_SECTIONS or pick_stages(datetime.date.today().toordinal(), 11.5, CUDA, STAGE_GROUPS["chat_image"])
os.environ["SHAM_ACTIVE_SECTIONS"] = ",".join(STAGES_TODAY)
install_section_gate()
print("الجهاز:", "GPU" if CUDA else "CPU", "| الأقسام التي تعمل اليوم:", STAGES_TODAY)


---
# القسم: المرحلة الثالثة — الدمج والمحادثة
`sham_chat_stage` — خلاياه أدناه كما هي في دفتره الأصلي.

In [ ]:
# SHAM_SECTION: sham_chat_stage
get_ipython().run_line_magic("reset", "-f")   # قسم جديد = دفتر جديد: لا متغيرات من قسم سابق
import gc

gc.collect()
try:
    import torch
    torch.cuda.empty_cache()   # يحرّر ذاكرة GPU بين قسمين في جلسة واحدة
except Exception:
    pass


# 💬 شام — مرحلة التجميع والمحادثة (المرحلة الثالثة)

هنا يصبح شام **نموذجاً واحداً كاملاً** يجيب بالنص والصورة والصوت والفيديو، ويبحث بنفسه عند الحاجة. كل تشغيل:

1. **الدمج المحروس للمعرفة**: يجمع ما تعلّمته كل مسارات شام (المرحلة الأولى، Track A، Track B البحث الذاتي، وأحدث المرحلة الثانية) في هذا الدماغ. يُقبل الدمج فقط إن تحسّنت كل المهارات معاً، وإلا تُعاد الأوزان كما كانت. ويدخل نص Track B المجموع من الإنترنت في «التذكّر».
2. **صيغة حوار شام**: `<BOS> <USER> سؤال <SHAM> جواب <EOS>`، فيتعلّم الإجابة ومتى يتوقف.
3. **كل الوسائط داخل المحادثة**:
   - الصورة: «صف هذه الصورة» ↔ «ارسم: …»
   - الصوت: «اكتب ما يقال» ↔ «انطق: …»
   - **الفيديو بصوته**: «صف هذا الفيديو» ↔ «أنشئ فيديو: …» (لأول مرة يتعلّم شام الفيديو فعلاً).
4. **البحث المتعلَّم**: يكتب شام بنفسه `<SEARCH_START> استعلام <SEARCH_END>`، فتصله النتيجة ويكمل منها. يتعلّم ذلك من ويكيبيديا العربية دون أي تسميات يدوية.
5. **التذكّر**: نص عربي ممزوج بالتدريب، حتى لا ينسى ما تعلّمه.
6. **المكافأة الذاتية المرتكزة**: يكتب عدة إجابات، ويتعلّم تفضيل أفضلها على أسوئها. تُقبل الجولة فقط إن لم يسؤ على أسئلة لم يرها.
7. **المقياس الثابت** قبل الجلسة وبعدها، ويُرسل إلى تيليجرام.

البيانات: `arbml/CIDAR` و`CohereLabs/aya_dataset` العربية (apache-2.0، مكتوبة بأيدٍ بشرية)، وويكيبيديا العربية، وصور وأصوات ومقاطع Kinetics المرخّصة بالمشاع الإبداعي فقط. كل تشغيل يأخذ بيانات **جديدة** ويكمل بعد آخر موضع.

**الإعداد:** Accelerator: أي خيار (GPU أو None). Internet: On. Secrets: `GITHUB_TOKEN`، `KAGGLE_USERNAME`، `KAGGLE_KEY` (واختيارياً `TELEGRAM_BOT_TOKEN` و`TELEGRAM_CHAT_ID`).

### 1) سحب كود شام من GitHub

In [ ]:
# SHAM_SECTION: sham_chat_stage
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
    print("تم سحب الكود الحقيقي من مستودع GitHub بنجاح.")
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
    print("الكود موجود بالفعل في هذه الجلسة — تم سحب أي تحديثات جديدة عليه.")

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py")), f"لم يتم العثور على model.py داخل {CODE_DIR}"
sys.path.insert(0, CODE_DIR)
print("كود ShamSmall الحقيقي جاهز في:", CODE_DIR)


In [ ]:
# SHAM_SECTION: sham_chat_stage
try:
    import tokenizers
    print(f"مكتبة tokenizers متوفرة مسبقاً (نسخة {tokenizers.__version__}).")
except ImportError:
    subprocess.run(["pip", "install", "-q", "tokenizers"], check=True)
    print("تم تثبيت مكتبة tokenizers.")

try:
    import soundfile
    print("مكتبة soundfile متوفرة مسبقاً.")
except ImportError:
    subprocess.run(["pip", "install", "-q", "soundfile"], check=True)
    print("تم تثبيت مكتبة soundfile.")


### 2) تحميل آخر نقطة لشام مع أداة تقسيم النص وأداتي الصورة والصوت المحفوظة معها

In [ ]:
# SHAM_SECTION: sham_chat_stage
import time
SESSION_START = time.time()
import torch
from pathlib import Path
from checkpoint import load_checkpoint
from text_tokenizer import ShamTextTokenizer
from sham_inputs import fetch_dataset
from tokenizer_select import select_pretrained_tokenizer
from train_image_tokenizer import save_tokenizer_checkpoint as save_image_tokenizer
from train_audio_tokenizer import save_tokenizer_checkpoint as save_audio_tokenizer
from sham_chat import load_progress

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"الجهاز: {device}")

CHAT_OWN = fetch_dataset("sham-chat-checkpoint")
_own_ck = sorted(CHAT_OWN.rglob("final_chat.pt")) if CHAT_OWN else []
_mm = fetch_dataset("sham-multimodal-checkpoint")
_mm_ck = sorted(_mm.rglob("final_" + "multimodal.pt")) if _mm else []
assert _own_ck or _mm_ck, "لم يُنشر ناتج المرحلة الثانية (sham-multimodal-checkpoint) بعد — شغّلي دفتر المرحلة الثانية أولاً."

# نقطة الانطلاق بالقياس لا بالعادة (قرار 2026-10-11): ناتج هذه المرحلة السابق كان بمستوى التخمين العشوائي (خسارة 10.7) لأنها تكمل دائماً
# من ناتجها؛ الآن يُقارَن بناتج المرحلة الثانية على نص ثابت بمُرمِّز كل منهما، ولا يستمر إلا إن لم يكن أسوأ بأكثر من 5%.
from sham_base_select import choose_base, make_probe
_cands = []
for _name, _ckl in (("chat", _own_ck), ("multimodal", _mm_ck)):
    if _ckl:
        _toks = sorted(_ckl[0].parent.rglob("sham_small_tokenizer.json"))
        _cands.append((_name, _ckl[0], _toks[0] if _toks else None))
_best = _cands[0][0]
if len(_cands) > 1:
    try:
        _best, _ = choose_base(_cands, make_probe(device), keep="chat")
    except Exception as _exc:
        print(f"⚠ تعذّر القياس ({type(_exc).__name__}): يُعتمد ناتج المرحلة الثانية")
        _best = "multimodal"
if _best == "chat":
    base_ckpt, BASE_DIR = _own_ck[0], _own_ck[0].parent
    print("استئناف مرحلة المحادثة من ناتجها السابق (الأفضل بالقياس).")
else:
    CHAT_OWN = None
    base_ckpt, BASE_DIR = _mm_ck[0], _mm_ck[0].parent
    print("بداية مرحلة المحادثة من ناتج المرحلة الثانية (الأفضل بالقياس).")

model, start_step, _ = load_checkpoint(base_ckpt, map_location=device)
tokenizer = ShamTextTokenizer.load(str(sorted(BASE_DIR.rglob("sham_small_tokenizer.json"))[0]))
tokenizer.save("/kaggle/working/sham_small_tokenizer.json")
print(f"نقطة الانطلاق: {base_ckpt.name} — الخطوة {start_step:,}، vocab={tokenizer.vocab_size:,}")

progress = load_progress([CHAT_OWN]) if CHAT_OWN else {}
progress.setdefault("merged", {})
if not CHAT_OWN:  # نقطة الانطلاق نفسها لا تُدمج مع نفسها
    progress["merged"]["sham-multimodal-checkpoint"] = start_step

# نفس أداتي الترميز اللتين تدرّب بهما النموذج (المحفوظتين مع النقطة نفسها).
_img, _aud = select_pretrained_tokenizer("image", BASE_DIR), select_pretrained_tokenizer("audio", BASE_DIR)
assert _img and _aud, f"أداتا الصورة/الصوت غير موجودتين بجانب {base_ckpt.name} — أعيدي تشغيل المرحلة الثانية لتنشرهما."
(image_tokenizer, _img_step, _), (audio_tokenizer, _aud_step, _) = _img, _aud
image_tokenizer, audio_tokenizer = image_tokenizer.to("cpu").train(False), audio_tokenizer.to("cpu").train(False)
save_image_tokenizer("/kaggle/working/image_tokenizer.pt", image_tokenizer, step=_img_step)
save_audio_tokenizer("/kaggle/working/audio_tokenizer.pt", audio_tokenizer, step=_aud_step)
from sham_media_link import confirm_saved_tokenizers
confirm_saved_tokenizers(image_tokenizer, audio_tokenizer, BASE_DIR)

### 3) بيانات هذه الجلسة: حوار + صورة + صوت + فيديو + بحث + نص للتذكّر (مع عيّنة تحقق لا يُدرَّب عليها)

In [ ]:
# SHAM_SECTION: sham_chat_stage
import json, random
import soundfile as sf
from PIL import Image
from model import image_token_id_to_vocab_id, audio_token_id_to_vocab_id
from mel_spectrogram import waveform_to_mel_spectrogram
from sham_chat import (load_arabic_dialogues, build_chat_example, build_media_chat_examples, build_video_chat_examples,
                       build_search_example, load_wiki_articles, split_holdout, batch_examples,
                       text_replay_batches, interleave)
from sham_merge import research_corpus_files
from sham_data_sources import Ledger, collect
from dataset import ImageCaptionDataset, AudioTranscriptDataset, MultimodalCollator, AudioMultimodalCollator

MAX_DIALOGUES = 4_000      # أمثلة حوار جديدة في كل جلسة
MAX_MEDIA = 600            # صور + مقاطع صوت (من كل نوع)
MAX_VIDEOS = 60            # مقاطع فيديو بصوتها (Kinetics المرخّصة CC فقط)
VIDEO_KEYFRAMES = 2        # إطارات كل فيديو داخل سياق النموذج (يطابق ما يطلبه البوت)
MAX_SEARCH = 1_500         # أمثلة «ابحث ثم أجب» من ويكيبيديا
CHAT_BATCH = 8
MAX_LEN = min(1024, model.cfg.max_seq_len)
rng = random.Random(start_step)
Path("/kaggle/working/checkpoints").mkdir(parents=True, exist_ok=True)

# — الحوار
dialogues, progress = load_arabic_dialogues(progress, MAX_DIALOGUES)
if not dialogues:  # انتهت كل الأمثلة المتاحة: دورة جديدة عليها
    progress.update({"cidar": 0, "aya_ar": 0, "epoch": int(progress.get("epoch", 0)) + 1})
    dialogues, progress = load_arabic_dialogues(progress, MAX_DIALOGUES)
train_dialogues, holdout_dialogues = split_holdout(dialogues, seed=start_step)
print(f"حوار: {len(train_dialogues):,} للتدريب + {len(holdout_dialogues):,} للتحقق (الدورة {progress.get('epoch', 0)})")
chat_examples = [build_chat_example(tokenizer.encode(q), tokenizer.encode(a), max_len=MAX_LEN) for q, a in train_dialogues]
holdout_chat = [build_chat_example(tokenizer.encode(q), tokenizer.encode(a), max_len=MAX_LEN) for q, a in holdout_dialogues]
# — محادثات متعددة الأدوار (2026-10-14): كل أدوار المحادثة، والخسارة على ردود شام فقط (sham_chat.build_multiturn_example). تُضاف للتدريب فقط؛
#   أمثلة التحقق (holdout_chat) وبوابة الدمج تبقى كما هي. إيقافها: SHAM_CHAT_MULTITURN=0
MAX_CONVERSATIONS = 1_000
try:
    from sham_chat import conversation_examples, load_arabic_conversations, multiturn_enabled
    if multiturn_enabled():
        _convs, progress = load_arabic_conversations(progress, MAX_CONVERSATIONS)
        _mt = conversation_examples(_convs, tokenizer.encode, max_len=MAX_LEN)
        chat_examples += _mt
        print(f"محادثات متعددة الأدوار للتدريب: {len(_mt):,}")
except ImportError as _exc:      # كود أقدم في الفرع المستنسَخ: تبقى المرحلة كما كانت
    print(f"⚠ المحادثات متعددة الأدوار غير متاحة في هذا الكود: {_exc}")
# — التصحيح الذاتي المتعلَّم: مسودة معطوبة بأخطاء شام نفسها ← الإجابة الصحيحة
from sham_correct import correction_examples
correct_examples = correction_examples(tokenizer.encode, train_dialogues, share=0.25, seed=start_step, max_len=MAX_LEN)
print(f"أمثلة التصحيح الذاتي: {len(correct_examples):,}")

# — الصورة والصوت داخل المحادثة
media_examples = []
from sham_media_link import collect_fixed_pairs, data_report
chat_ledgers = {k: Ledger.load(k, name=f"chat_{k}") for k in ("image", "audio")}
FIXED_PAIRS = collect_fixed_pairs(("image", "audio"), chat_ledgers, image_size=image_tokenizer.cfg.image_size)
for kind in ("image", "audio"):
    ledger = chat_ledgers[kind]
    manifest, _stats = collect(kind, f"/kaggle/working/corpus/{kind}", MAX_MEDIA, ledger, image_size=image_tokenizer.cfg.image_size)
    ledger.save("/kaggle/working/checkpoints")
    import sham_media_shards   # كل المصادر معاً: المباشر + عيّنة من «مصنع الوسائط»
    manifest = sham_media_shards.with_factory(kind, manifest, f"/kaggle/working/corpus/{kind}_factory", k=1, limit=1_500) or manifest
    if kind == "image":
        ds = ImageCaptionDataset(manifest, image_size=image_tokenizer.cfg.image_size)
        encode = MultimodalCollator(tokenizer, image_tokenizer).encode_batch_images
    else:
        ds = AudioTranscriptDataset(manifest, n_mels=audio_tokenizer.cfg.n_mels, segment_frames=audio_tokenizer.cfg.segment_frames, max_segments=4)
        encode = AudioMultimodalCollator(tokenizer, audio_tokenizer).encode_batch_audio
    data_report("الصورة" if kind == "image" else "الصوت", _stats, ds)
    for s in range(0, len(ds), 16):
        items = [ds[i] for i in range(s, min(s + 16, len(ds)))]
        with torch.no_grad():
            ids = encode(items)
        for (caption, _), media in zip(items, ids):
            media_examples += build_media_chat_examples(tokenizer.encode, caption, media.tolist(), kind, rng)
media_examples, holdout_media = split_holdout(media_examples, seed=start_step + 1)
print(f"صورة/صوت داخل المحادثة: {len(media_examples):,} (+ {len(holdout_media)} للتحقق)")

# — الفيديو بصوته (لأول مرة داخل النموذج نفسه)
def _frame_tensor(path):
    size = image_tokenizer.cfg.image_size
    im = Image.open(path).convert("RGB").resize((size, size))
    return torch.tensor(list(im.getdata()), dtype=torch.float32).view(size, size, 3).permute(2, 0, 1) / 127.5 - 1.0

video_examples = []
try:
    video_ledger = Ledger.load("video", name="chat_video")
    video_manifest, _ = collect("video", "/kaggle/working/corpus/videos", MAX_VIDEOS, video_ledger,
                                image_size=image_tokenizer.cfg.image_size, num_frames=8)
    video_ledger.save("/kaggle/working/checkpoints")
    vroot = Path(video_manifest).parent
    for line in open(video_manifest, encoding="utf-8"):
        rec = json.loads(line)
        paths = rec["frames"]
        picks = [paths[round(i * (len(paths) - 1) / max(VIDEO_KEYFRAMES - 1, 1))] for i in range(VIDEO_KEYFRAMES)]
        with torch.no_grad():
            frames = torch.stack([_frame_tensor(vroot / p) for p in picks])
            grids = image_tokenizer.encode(frames)
            frame_ids = image_token_id_to_vocab_id(grids.view(len(picks), -1)).tolist()
            audio_ids = None
            if rec.get("audio"):
                wave, sr = sf.read(str(vroot / rec["audio"]), dtype="float32", always_2d=False)
                wave = torch.from_numpy(wave.mean(axis=1) if wave.ndim > 1 else wave)
                from sham_media_codec import audio_vocab_ids   # كل مقاطع الصوت (حتى 3)، لا أول 2.56 ثانية فقط (إصلاح 2026-10-11)
                audio_ids = audio_vocab_ids(audio_tokenizer, wave, sr, max_segments=3)
        video_examples += build_video_chat_examples(tokenizer.encode, rec["caption"], frame_ids, audio_ids, rng)
except Exception as exc:  # الفيديو لا يوقف بقية الجلسة
    print(f"⚠ تعذّر جمع الفيديو هذه الجلسة: {exc}")
print(f"فيديو داخل المحادثة: {len(video_examples):,}")

# — البحث المتعلَّم (ابحث ثم أجب) من ويكيبيديا العربية
search_examples = []
try:
    _skip = int(progress.get("wiki_search", 0))
    for title, text in load_wiki_articles(MAX_SEARCH, _skip):
        ex = build_search_example(tokenizer.encode, title, text, rng)
        if ex and len(ex[0]) <= MAX_LEN:
            search_examples.append(ex)
    progress["wiki_search"] = _skip + MAX_SEARCH
except Exception as exc:
    print(f"⚠ تعذّر تجهيز أمثلة البحث: {exc}")
print(f"أمثلة «ابحث ثم أجب»: {len(search_examples):,}")

# — التذكّر: ويكيبيديا + نص Track B المجموع من الإنترنت (دمج المعرفة على مستوى البيانات)
_research = fetch_dataset("sham-research-track-checkpoint-v2")
extra_text = research_corpus_files(_research)
print(f"نص البحث الذاتي (Track B) المضاف للتذكّر: {len(extra_text)} ملف")
chat_batches = batch_examples(chat_examples, CHAT_BATCH, seed=start_step)
text_batches = text_replay_batches(tokenizer, min(512, MAX_LEN), 4, max(len(chat_batches) // 2, 1) + 4,
                                   "/kaggle/working/replay_text", max_position=20_000, seed=start_step,
                                   extra_files=extra_text)
holdout_text, text_batches = text_batches[:4], text_batches[4:]
media_batches = batch_examples(media_examples, 4, seed=start_step + 1)
video_batches = batch_examples(video_examples, 2, seed=start_step + 2)
search_batches = batch_examples(search_examples, 4, seed=start_step + 3)
correct_batches = batch_examples(correct_examples, CHAT_BATCH, seed=start_step + 4)
# — التفكير المتسلسل: حلول مكتوبة خطوة بخطوة (مولَّدة ومفحوصة آلياً). الحصة تقرّرها شام بنفسها من مسبار تقيسه بعد كل دمج
#   (sham_auto_config) — أو تُفرض/تُمنع بمتغير SHAM_REASONING_SHARE. بلا قرار بعد: 0 فلا شيء يتغيّر.
from sham_reasoning_data import reasoning_examples, share as _reasoning_share
_rs = _reasoning_share()
_n_other = len(chat_batches) + len(media_batches) + len(video_batches) + len(search_batches) + len(text_batches) + len(correct_batches)
reason_batches = []
if _rs > 0:
    reason_batches = batch_examples(reasoning_examples(tokenizer.encode, int(_n_other * _rs / (1 - _rs)) * CHAT_BATCH, seed=start_step + 5,
                                                       max_len=min(384, MAX_LEN)), CHAT_BATCH, seed=start_step + 5)
print(f"تفكير متسلسل: {len(reason_batches):,} دفعة (الحصة المقرَّرة {_rs:.0%})")
all_batches = interleave([chat_batches, media_batches, video_batches, search_batches, text_batches, correct_batches, reason_batches],
                         [1.0, 0.6, 0.3, 0.4, 0.5, 0.3, (3.1 * _rs / (1 - _rs)) if _rs > 0 else 0.0], seed=start_step)
# ميزان الحارس: عيّنات لم يُدرَّب عليها من كل مهارة (محدودة الحجم ليبقى سريعاً)
GATE = {"chat": batch_examples(holdout_chat, 8)[:8], "media": batch_examples(holdout_media, 4)[:6], "text": holdout_text}
GATE = {k: v for k, v in GATE.items() if v}
print(f"دفعات: حوار {len(chat_batches):,} + صورة/صوت {len(media_batches):,} + فيديو {len(video_batches):,} "
      f"+ بحث {len(search_batches):,} + تذكّر {len(text_batches):,} + تصحيح {len(correct_batches):,} = {len(all_batches):,}")
assert all_batches, "لا توجد بيانات لهذه الجلسة."

### 4) المقياس الثابت — قبل الجلسة

In [ ]:
# SHAM_SECTION: sham_chat_stage
from sham_eval import evaluate, fixed_eval_pairs, format_eval

EVAL_PAIRS = fixed_eval_pairs()
model.to(device)
eval_before = evaluate(model, tokenizer, device, chat=CHAT_OWN is not None, pairs=EVAL_PAIRS)
print(format_eval("قبل الجلسة", start_step, eval_before))
from sham_media_link import measure_link, link_report
_media_toks = {"image": image_tokenizer, "audio": audio_tokenizer}
link_before = measure_link(model, tokenizer, _media_toks, FIXED_PAIRS, device, chat=True)
link_report(link_before)

### 5) الدمج المحروس للمعرفة

يُجرَّب كل مسار بنسب مختلفة، ويُقبل فقط إن تحسّنت **كل** المهارات (حوار، صورة/صوت، نص) على بيانات لم يُدرَّب عليها. كل نسخة من كل مسار تُدمج مرة واحدة فقط.

In [ ]:
# SHAM_SECTION: sham_chat_stage
from sham_merge import KNOWN_SOURCES, load_source, guarded_merge, batches_loss

def gate_scores(m):
    return {k: batches_loss(m, v, device) for k, v in GATE.items()}

merge_sources = []
merge_report = []
for _name, _pattern, _rows in KNOWN_SOURCES:
    _root = fetch_dataset(_name)
    if not _root:
        continue
    _loaded, _why = load_source(_root, _pattern, model, tokenizer)
    if _loaded is None:
        merge_report.append(f"⏭ {_name}: {_why}")
        continue
    _other, _step = _loaded
    if progress["merged"].get(_name, -1) >= _step:
        merge_report.append(f"⏭ {_name}: الخطوة {_step:,} جُرّبت سابقاً")
        continue
    merge_sources.append((f"{_name} (خطوة {_step:,})", _other.to(device), _rows))
    progress["merged"][_name] = _step

if merge_sources:
    merge_report += guarded_merge(model, merge_sources, gate_scores)
del merge_sources
print("\n".join(merge_report) or "لا مسارات جديدة للدمج هذه الجلسة.")

### 6) التدريب (على أي معالج تم اختياره، ضمن وقت الجلسة)

In [ ]:
# SHAM_SECTION: sham_chat_stage
from train import TrainConfig, train, build_optimizer

GRAD_ACCUM = 2
SESSION_HOURS = 5.0 if device.startswith("cuda") else 6.0   # حد الجلسة (GPU أقصر حفاظاً على الرصيد)
RESERVE_SECONDS = 50 * 60                                     # للمكافأة الذاتية والمقياس والنشر
optimizer = build_optimizer(model, lr=5e-5, weight_decay=0.1)

t0 = time.time()
model.train()
for _b in all_batches[:5]:  # قياس السرعة الحقيقية على هذا الجهاز (لا تحديث للأوزان)
    ids, labels = _b if isinstance(_b, tuple) else (_b, _b)
    _, loss = model(ids.to(device), labels=labels.to(device))
    loss.backward()
optimizer.zero_grad()
per_batch = max((time.time() - t0) / min(5, len(all_batches)), 1e-3)
train_seconds = max(600, SESSION_HOURS * 3600 - (time.time() - SESSION_START) - RESERVE_SECONDS)
session_batches = all_batches[: int(train_seconds * 0.9 / per_batch)]
TOTAL_STEPS = max(len(session_batches) // GRAD_ACCUM, 1)
print(f"~{per_batch:.2f} ث/دفعة على {device} — {len(session_batches):,} دفعة = {TOTAL_STEPS:,} خطوة تحديث.")

train_cfg = TrainConfig(
    seq_len=MAX_LEN, batch_size=CHAT_BATCH, grad_accum_steps=GRAD_ACCUM, lr=5e-5,
    warmup_steps=max(10, TOTAL_STEPS // 50), total_steps=start_step + TOTAL_STEPS,
    checkpoint_dir="/kaggle/working/checkpoints", checkpoint_every=200, log_every=20,
    max_wall_clock_seconds=train_seconds,
)
loss_history = train(model, session_batches, train_cfg, device=device, start_step=start_step, resume_optimizer=optimizer)
final_step = start_step + len(loss_history)
print(f"انتهى: {len(loss_history):,} خطوة — الخطوة النهائية {final_step:,}")

### 7) المكافأة الذاتية المرتكزة (تُقبل فقط إن لم يسؤ على أسئلة لم يرها)

In [ ]:
# SHAM_SECTION: sham_chat_stage
from sham_reward import guarded_self_reward

REWARD_QUESTIONS = 48
_qa = random.Random(final_step).sample(train_dialogues, min(REWARD_QUESTIONS, len(train_dialogues)))
reward_report = guarded_self_reward(model, tokenizer, _qa, device,
                                    gate_fn=lambda m: sum(gate_scores(m).values()))
print(reward_report)

# الاستدلال المتحقَّق (GRPO): يقيس أولاً، ويبقى نائماً حتى تتجاوز إصابة شام العتبة
from sham_grpo import guarded_grpo
grpo_report = guarded_grpo(model, tokenizer, device, gate_fn=lambda m: sum(gate_scores(m).values()), seed=final_step)
print(grpo_report)

# هل تعلّم شام إصلاح مسودته؟ (خسارة الإجابة بلا مسودة مقابل بعد مسودة معطوبة)
from sham_correct import revision_gain, format_gain
correct_gain = revision_gain(model, tokenizer.encode, holdout_dialogues[:32], device, seed=final_step)
correct_report = format_gain(correct_gain)
print(correct_report)

### 8) المقياس الثابت — بعد الجلسة، وتقرير تيليجرام

In [ ]:
# SHAM_SECTION: sham_chat_stage
eval_after = evaluate(model, tokenizer, device, chat=True, pairs=EVAL_PAIRS)
link_summary = link_report(link_before, measure_link(model, tokenizer, _media_toks, FIXED_PAIRS, device, chat=True))
report = "\n".join([format_eval("مرحلة التجميع والمحادثة", final_step, eval_after, before=eval_before),
                    link_summary, "🧩 دمج المعرفة:", *(merge_report or ["لا جديد"]), reward_report, grpo_report, correct_report])
print(report)
try:
    from kaggle_secrets import UserSecretsClient
    _tg_token = UserSecretsClient().get_secret("TELEGRAM_BOT_TOKEN")
    _tg_chat = UserSecretsClient().get_secret("TELEGRAM_CHAT_ID")
except Exception:
    _tg_token, _tg_chat = None, None
from telegram_report import send_telegram_message
send_telegram_message(_tg_token, _tg_chat, report[:4000])

### 9) الحفظ والنشر إلى `sham-chat-checkpoint`

In [ ]:
# SHAM_SECTION: sham_chat_stage
import shutil
from checkpoint import save_checkpoint
from sham_inputs import publish_dataset

save_checkpoint("/kaggle/working/checkpoints/final_chat.pt", model, final_step)
upload_dir = Path("/kaggle/working/for_dataset_upload")
if upload_dir.exists():
    shutil.rmtree(upload_dir)
upload_dir.mkdir(parents=True)
shutil.copy2("/kaggle/working/checkpoints/final_chat.pt", upload_dir / "final_chat.pt")
for _f in ("sham_small_tokenizer.json", "image_tokenizer.pt", "audio_tokenizer.pt"):
    shutil.copy2(Path("/kaggle/working") / _f, upload_dir / _f)
for _ledger in Path("/kaggle/working/checkpoints").glob("*_ledger.json"):
    shutil.copy2(_ledger, upload_dir / _ledger.name)
(upload_dir / "chat_progress.json").write_text(json.dumps(progress), encoding="utf-8")
(upload_dir / "eval_history.json").write_text(json.dumps(
    {"step": final_step, "before": {k: v for k, v in eval_before.items() if k != "samples"},
     "after": {k: v for k, v in eval_after.items() if k != "samples"},
     "merge": merge_report, "reward": reward_report, "grpo": grpo_report, "correct": correct_gain}, ensure_ascii=False), encoding="utf-8")
publish_dataset(upload_dir, "sham-chat-checkpoint", f"chat stage at step {final_step:,}")

---
# القسم: مسار ترميز الصورة (ثم الصوت والفيديو)
`sham_image_tokenizer_track` — خلاياه أدناه كما هي في دفتره الأصلي.

In [ ]:
# SHAM_SECTION: sham_image_tokenizer_track
get_ipython().run_line_magic("reset", "-f")   # قسم جديد = دفتر جديد: لا متغيرات من قسم سابق
import gc

gc.collect()
try:
    import torch
    torch.cuda.empty_cache()   # يحرّر ذاكرة GPU بين قسمين في جلسة واحدة
except Exception:
    pass


# شام — مسار جمع وتدريب أداة ترميز الصورة (VQ-VAE) — CPU، بلا حاجة لـ GPU

نظير دفتر الصوت تماماً، لكن للصورة. **نطاق حقيقي يجب معرفته بصراحة:** تعليقات الصور (captions) العربية الأصلية شحيحة — لكن هذا لا يهم في هذا الدفتر تحديداً: تدريب أداة ترميز الصورة نفسها (VQ-VAE) **لا يحتاج تعليقات إطلاقاً** (تعلّم غير مُشرَف بالكامل، إعادة بناء بكسلات فقط) — التعليقات تُستخدم لاحقاً فقط في دفتر التدريب متعدد الوسائط (فهم/توليد الصورة من نص). لذلك هذا الدفتر يجمع صوراً عربية حقيقية (مع تعليقاتها إن وُجدت، تُحفظ للاستخدام لاحقاً) لكنه يتدرّب على البكسلات وحدها.

يحاول أولاً مصدراً عربياً حقيقياً (`Misraj/Arabic-Image-Captioning_100M`)، ويتراجع تلقائياً لـ Flickr30k (تعليقات إنجليزية، لكن الصور نفسها حقيقية ومتنوعة وهذا كل ما يلزم لهذه المرحلة) إن فشل المصدر العربي — تحقّق حقيقي وقت التشغيل وليس افتراضاً، تماماً كما في دفتر المرحلة الثانية.

كل تشغيل: يجمع دفعة **جديدة** من الصور الحقيقية لم تُستخدم سابقاً، يكمل تدريب نفس أداة الترميز، وينشرها كمجموعة بيانات Kaggle خاصة بهذا المسار — جاهزة ليستخدمها لاحقاً دفتر التدريب متعدد الوسائط.

## قبل "Save Version → Save & Run All":
1. **فعّل الإنترنت** من Settings (لا حاجة لـ GPU).
2. تأكد من وجود نفس أسرار Kaggle: `GITHUB_TOKEN`، `KAGGLE_USERNAME`، `KAGGLE_KEY`.
3. **لا حاجة لإضافة أي Input**: الاستئناف من آخر نسخة من `sham-image-tokenizer-checkpoint` تلقائي (انظر الملاحظة أدناه).
4. استخدم **Save Version → Save & Run All (Commit)** دائماً، ويمكن جدولته للتشغيل التلقائي اليومي.


> **الاستئناف تلقائي بالكامل (2026-09-24):** لا حاجة لأي «Add Input». في كل تشغيل يجلب الدفتر آخر نسخة من مجموعة بياناته هو عبر Kaggle API (`sham_inputs.py`) إن لم تكن مرفقة، ويكمل منها. إن كانت مرفقة يدوياً تُستخدم كما هي.

### 1) سحب الكود الحقيقي من GitHub

In [ ]:
# SHAM_SECTION: sham_image_tokenizer_track
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py"))
sys.path.insert(0, CODE_DIR)
print("كود شام الحقيقي جاهز في:", CODE_DIR)


### 2) تثبيت المكتبات الإضافية

In [ ]:
# SHAM_SECTION: sham_image_tokenizer_track
try:
    import datasets
    print(f"مكتبة datasets متوفرة مسبقاً (نسخة {datasets.__version__}).")
except ImportError:
    subprocess.run(["pip", "install", "-q", "datasets"], check=True)
    print("تم تثبيت مكتبة datasets.")


### 3) استئناف أداة ترميز الصورة من التشغيل السابق (إن وُجد)

نفس المبدأ تماماً — حجم **إنتاجي كامل** (256×256، 8192 رمزاً، القيم الافتراضية في `image_tokenizer.py`) وليس نسخة مصغّرة، لأن هذا المسار مخصص لبنائها تدريجياً عبر تشغيلات غير محدودة.

In [ ]:
# SHAM_SECTION: sham_image_tokenizer_track
import json as _json
from pathlib import Path
from image_tokenizer import ImageTokenizer, ImageTokenizerConfig
from tokenizer_select import select_pretrained_tokenizer, _progress_samples

# الاستئناف: يُفحص كل ما في المدخلات ويُختار الأكثر تدريباً؛ النسخ غير المتوافقة
# (قاموس بحجم مختلف، أو محفوظة بإصدار أقدم) تُصلَح وتُقبل ولا تُتجاهل (tokenizer_select.py).
from sham_inputs import fetch_dataset

# آخر نسخة من مجموعة هذا المسار (أداة الترميز + progress + سجل البيانات): مرفقة، أو تُنزَّل
# تلقائياً عبر Kaggle API — فيستأنف دائماً حتى بلا أي «Add Input».
fetch_dataset("sham-image-tokenizer-checkpoint")

samples_consumed = 0
start_step = 0
print("البحث عن نسخة سابقة من أداة الترميز (image):")
_picked = select_pretrained_tokenizer("image")
if _picked:
    image_tokenizer, start_step, _resume_path = _picked
    image_tokenizer_cfg = image_tokenizer.cfg
    samples_consumed = _progress_samples(_resume_path, "image")
    print(f"استؤنف من: {_resume_path} (خطوة {start_step:,}، {samples_consumed:,} عيّنة سابقاً)")
else:
    image_tokenizer_cfg = ImageTokenizerConfig()
    image_tokenizer = ImageTokenizer(image_tokenizer_cfg)
    print("لا توجد نسخة سابقة صالحة — بدء أداة ترميز جديدة (متوقَّع فقط في أول تشغيل).")

### 4) جمع دفعة جديدة من صور حقيقية (عربية أولاً، Flickr30k عند الفشل)

`skip=samples_consumed` يضمن رؤية صور **جديدة** كل تشغيل.

In [ ]:
# SHAM_SECTION: sham_image_tokenizer_track
from sham_data_sources import Ledger, collect

# جمع هذا الدفتر المباشر من مصادره. الدفتر مفتوح للمهندس: يعدّل مصادر الجمع ويضيف خلايا. لذلك لا يُدمج «مصنع الوسائط» (Lightning) هنا، بل في خلية
# التدريب التالية: فيعمل أيّاً كان ما أنتجته خلية الجمع (خليتنا هذه أو خلية المهندس) ما دامت تضع الـ manifest في المتغير image_manifest.
MAX_IMAGES = 1_000            # حصة الجمع المباشر لكل جلسة
image_ledger = Ledger.load("image")
_direct_manifest, collect_stats = collect("image", "/kaggle/working/corpus/image_direct", MAX_IMAGES, image_ledger, image_size=image_tokenizer_cfg.image_size)
NEW_DATA = collect_stats.written > 0
image_manifest = _direct_manifest if NEW_DATA else None
print(f"جمع مباشر: {collect_stats.written:,} عيّنة جديدة | جاهز: {image_manifest}")


### 5) قياس سرعة CPU الحقيقية ثم التدريب ضمن ميزانية زمنية محددة

In [ ]:
# SHAM_SECTION: sham_image_tokenizer_track
import torch
import sham_media_shards
from sham_vq_train import train_stream
from sham_media_cfg import IMAGE_RES_MIX

# مصنع الوسائط ينضم هنا مهما كانت خلية الجمع التي سبقت (خليتنا أو خلية المهندس): يُدمج ما لم يُستهلك من دفعاته مع ما جُمع قبله.
image_manifest, NEW_DATA, FACTORY_TAKEN, _consumed = sham_media_shards.attach_factory(
    "image", globals().get("image_manifest"), "/kaggle/working/corpus/image_factory", 6)

if NEW_DATA:
    # مدرّب جديد (sham_vq_train.py): قراءة كسولة (أي عدد بذاكرة ثابتة)، قاموس يُبدأ من مخرجات المُشفِّر الحقيقية، أرموز ميتة تُحيا،
    # جدول معدل تعلّم، عدّاد خطوات حقيقي، ويستغل الوقت كله في التدريب.
    MAX_TRAINING_MINUTES = 150 if FACTORY_TAKEN else 45
    BATCH_SIZE = 16 if torch.cuda.is_available() else 4
    stats = train_stream(image_tokenizer, image_manifest, minutes=MAX_TRAINING_MINUTES, batch_size=BATCH_SIZE, workers=2, res_mix=IMAGE_RES_MIX)
    print(stats.summary())
else:
    stats = None
    print('تخطّي التدريب: لا بيانات جديدة.')


### 6) حفظ ونشر أداة ترميز الصورة (كمجموعة بيانات Kaggle خاصة بهذا المسار)

In [ ]:
# SHAM_SECTION: sham_image_tokenizer_track
from train_image_tokenizer import save_tokenizer_checkpoint

# كل ما يُقرأ من الخلايا السابقة له قيمة افتراضية: خلية أضافها المهندس أو عدّلها لا تُسقِط الحفظ.
_stats = globals().get("stats")
_collect = globals().get("collect_stats")
trained_steps = getattr(_stats, "steps", 0) if _stats else 0
final_step = globals().get("start_step", 0) + trained_steps
trained_samples = getattr(_stats, "samples", 0) if _stats else 0
final_samples_consumed = globals().get("samples_consumed", 0) + trained_samples

ckpt_dir = Path("/kaggle/working/checkpoints")
ckpt_dir.mkdir(parents=True, exist_ok=True)
save_tokenizer_checkpoint(ckpt_dir / "image_tokenizer.pt", image_tokenizer, step=final_step)
(ckpt_dir / "image_tokenizer_progress.json").write_text(
    _json.dumps({"samples_consumed": final_samples_consumed, "step": final_step,
                 "last_run_sources": (getattr(_collect, "per_source", None) or {"factory": trained_samples}),
                 "codebook_usage": getattr(_stats, "codebook_usage", None),
                 "recon": (_stats.recon[-1] if _stats and getattr(_stats, "recon", None) else None)})
)
(ckpt_dir / "image_factory_consumed.json").write_text(_json.dumps(sorted(set(globals().get("_consumed", set())) | set(globals().get("FACTORY_TAKEN", [])))))
_ledger = globals().get("image_ledger")
if _ledger is not None:
    _ledger.save(ckpt_dir)  # مواضع الاستئناف + بصمات كل ما جُمع — يرافق نقطة الحفظ دائماً
print(f"تم الحفظ عند الخطوة {final_step:,} (إجمالي العيّنات: {final_samples_consumed:,}) | "
      + (_collect.summary() if hasattr(_collect, "summary") else f"دفعات المصنع: {len(globals().get('FACTORY_TAKEN', []))}"))


In [ ]:
# SHAM_SECTION: sham_image_tokenizer_track
import json as _json
import shutil as _shutil

subprocess.run(["pip", "install", "-q", "-U", "kaggle"], check=False)

KAGGLE_USERNAME = UserSecretsClient().get_secret("KAGGLE_USERNAME")
KAGGLE_KEY = UserSecretsClient().get_secret("KAGGLE_KEY")
DATASET_SLUG = f"{KAGGLE_USERNAME}/sham-image-tokenizer-checkpoint"

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

upload_dir = Path("/kaggle/working/for_dataset_upload")
if upload_dir.exists():
    _shutil.rmtree(upload_dir)
upload_dir.mkdir(parents=True)
_shutil.copy2("/kaggle/working/checkpoints/image_tokenizer.pt", upload_dir / "image_tokenizer.pt")
_shutil.copy2("/kaggle/working/checkpoints/image_tokenizer_progress.json", upload_dir / "image_tokenizer_progress.json")
_shutil.copy2("/kaggle/working/checkpoints/image_ledger.json", upload_dir / "image_ledger.json")
_shutil.copy2("/kaggle/working/checkpoints/image_factory_consumed.json", upload_dir / "image_factory_consumed.json")

metadata = {"title": "sham-image-tokenizer-checkpoint", "id": DATASET_SLUG, "licenses": [{"name": "unknown"}]}
(upload_dir / "dataset-metadata.json").write_text(_json.dumps(metadata))

# نفس الفحص الحتمي المعتمد في كل الدفاتر الأخرى: نسأل Kaggle مباشرة هل
# مجموعة البيانات هذه موجودة أصلاً بدل تخمين ذلك من نص رسالة خطأ (غير
# موثّق وقد يتغيّر) — هذا الأسلوب هو ما كشف خطأ "-r skip" الصامت سابقاً.
from sham_inputs import dataset_exists as _dataset_exists_fn   # reads every page of the list (2026-10-10: page 1 only hid older datasets)
_dataset_exists = _dataset_exists_fn(DATASET_SLUG)

# "-r zip" وليس "skip": القيمة الافتراضية الموثّقة لـ Kaggle CLI للمجلدات
# الفرعية هي تجاهلها بصمت، لا رفعها — درس مستفاد من خطأ حقيقي سابق أثّر
# على كل دفاتر التدريب النصية قبل اكتشافه.
if _dataset_exists:
    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(upload_dir), "-m", f"auto-update at step {final_step:,}", "-r", "zip"],
        capture_output=True, text=True,
    )
else:
    result = subprocess.run(
        ["kaggle", "datasets", "create", "-p", str(upload_dir), "-r", "zip"],
        capture_output=True, text=True,
    )
_combined = (result.stdout or "") + (result.stderr or "")

if result.returncode == 0 and "error" not in _combined.lower():
    verb = "تم النشر إلى" if _dataset_exists else "تم إنشاء"
    print(f"{verb} {DATASET_SLUG} — التشغيل المجدول القادم سيلتقطها تلقائياً ويكمل من حيث توقفنا.")
elif "incompatible" in _combined.lower():
    print(
        "تحذير: هذه المجموعة أُنشئت سابقاً بصيغة رفع غير متوافقة. لا يوجد إصلاح على مستوى الكود لها تحديداً — "
        "غيّر الاسم أعلاه لاسم لم يُستخدم من قبل (مثلاً أضف -v2) وأعد التشغيل. نقطة الحفظ نفسها آمنة "
        "في Output هذه الجلسة بغض النظر."
    )
    print(_combined)
else:
    print("تحذير: فشل النشر — نقطة الحفظ لا تزال آمنة في Output هذه الجلسة. تأكد من صحة KAGGLE_USERNAME/KAGGLE_KEY.")
    print(_combined)

# ⛓ سلسلة (2026-10-12): بعد نشر أداة ترميز الصورة يكمل هذا الدفتر نفسه مسار الصوت ثم الفيديو في الجلسة ذاتها
# (الفيديو يحتاج أداتَي الصورة والصوت المنشورتين للتو، فالترتيب صورة ← صوت ← فيديو). ثلاثة مسارات قصيرة في مقعد واحد
# بدل ثلاثة مقاعد؛ فيُلغى جدول دفتري الصوت والفيديو ويتحرر مقعدان للمسارات الطويلة. فشل مسار لا يوقف التالي.
try:
    from sham_chain import run_notebooks
    CHAIN_RESULTS = run_notebooks(["sham_audio_tokenizer_track", "sham_video_tokenizer_track"], code_dir=CODE_DIR)
    print("نتيجة السلسلة:", CHAIN_RESULTS)
except Exception as _chain_exc:
    print("تحذير: تعذّر تشغيل السلسلة (الصورة نفسها محفوظة ومنشورة):", _chain_exc)


In [ ]:
# Ask the restarter to run right after this session ends (added 2026-10-16; deployment only, no training change).
try:
    import sham_trigger_restarter
    sham_trigger_restarter.trigger()
except Exception as _exc:
    print(f"sham_trigger_restarter skipped: {_exc}")
